# luc3d Tracker (Python port) — frame 0 + frame 1 demo

Python re-implementation of the algorithm behind LUCID's **Track All** button (`luc3d/pose/tracker.js`), built to run against a live `lucid-lite` `Session`.

**What's in here**
- §0 Launch lucid-lite (GUI window)
- §1 Frame 0 — *cold-start*: pairwise epipolar Hungarian → reproj-OKS refinement → graft remaining cameras → triangulate → assign `id_0..id_{N-1}` in descending cross-view confidence order
- §2 Frame 1 — *with priors*: same pipeline plus the 4-signal `reorder_groups_by_prev` Hungarian that keeps identity labels stable across frames
- §3 Propagation table — `(cam, track) → identity_id` before vs after frame 1
- §4 Optional: run a small range, then `track_all` over the whole session

**Prereqs**
- `scipy` (Hungarian via `linear_sum_assignment`)
- `cv2` is *optional* — used for distortion-aware undistort; falls back to identity when missing
- Helper functions live in `luc3d_tracker_helper.py` next to this notebook

## 0. Launch lucid-lite

Returns `(app, window)`; we also bind `session` for convenience. **Keep all three in scope** for the lifetime of the kernel.

In [ ]:
%gui qt
import sys
from pathlib import Path

GUI_SRC = str(Path.cwd() / "gui_source")
if GUI_SRC not in sys.path:
    sys.path.insert(0, GUI_SRC)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import main
import luc3d_tracker_helper as lt

# Pass a folder to skip the picker:
FOLDER = "/Users/joshuapark/Documents/talmolab/lucid_folders/10072022145420_small"
app, window = main.main(["main.py", FOLDER])
session = window.session

print("cameras:", session.camera_names())
print("frames :", session.min_frame, "->", session.max_frame,
      f"({len(session.frame_groups)} populated)")
print("tracks :", session.tracks)
print("cv2 available:", lt._HAS_CV2)

## 1. Frame 0 — cold start

No `prev_assignments`, no `prev_targets3d`. The algorithm:

1. `collect_instances(fg, cameras)` — flatten linked + unlinked per cam
2. `match_pairwise(...)`
   - sort cams by candidate count → bootstrap pair = (cam1, cam2)
   - epipolar score matrix → Hungarian on `-score`
   - refine top matches with `cross_view_score = 0.4·epipolar + 0.6·reproj-OKS`
   - cut: top-`num_animals` (or `score > 0.05` unconstrained)
   - graft remaining cams via reprojection-distance Hungarian (cutoff 100 px)
3. `triangulate_group(...)` → list of 3D keypoints per group → `targets3d`
4. Identity assignment: vote phase is skipped (no priors), so groups in descending-confidence order pick the first unused `id_i` from the pool.

Side-effects on `session`: identities `id_0..id_{N-1}` get created (if not present), and `session.track_identity_map` + `session.frame_identity_map` get written. The GUI repaints automatically (signals are emitted on every write).

We pass a single `TrackerCaches` instance through both frames so the F-matrix cache survives — same speed-up as in JS.

In [ ]:
import luc3d_tracker_helper as lt
import importlib; importlib.reload(lt)  # in case you edit the helper

# Wipe state so the run is reproducible (mirrors what trackAll does on entry)
session.identities = []
session.track_identity_map = {}
session.frame_identity_map = {}
session._identity_counter = 0
session.identities_changed.emit()
session.identity_map_changed.emit()

NUM_ANIMALS = 4   # set to None for unconstrained
FRAME0 = session.min_frame

caches = lt.TrackerCaches()
fg0 = session.frame_group(FRAME0)

r0 = lt.match_frame_instances(
    fg0, session.cameras, session,
    num_animals=NUM_ANIMALS,
    prev_assignments=None,
    prev_targets3d=None,
    per_frame=True,
    caches=caches,
)

print(f"frame {FRAME0}")
print(f"  groups found:      {len(r0['groups'])}")
print(f"  group sizes:       {[len(g) for g in r0['groups']]}  (cams per group)")
print(f"  identities created: {[(i.id, i.name, i.color) for i in session.identities]}")
print(f"  assignments (n={len(r0['assignments'])}):")
for k, v in sorted(r0['assignments'].items()):
    print(f"    {k:>10s} -> id={v}")

### Visualize frame 0 — every camera, overlay coloured by **identity**

Flip the live GUI's color mode to *identity* so the on-screen overlays match what you see here:

In [ ]:
session.set_color_mode("identity")
window.set_current_frame(FRAME0)

import math
import numpy as np
import matplotlib.pyplot as plt
from PySide6.QtGui import QImage

def qimage_to_numpy(qimg: QImage) -> np.ndarray:
    img = qimg.convertToFormat(QImage.Format_RGB888)
    w, h = img.width(), img.height()
    ptr = img.constBits()
    arr = np.frombuffer(ptr, dtype=np.uint8, count=img.sizeInBytes())
    arr = arr.reshape(h, img.bytesPerLine())[:, : w * 3]
    return arr.reshape(h, w, 3).copy()

def color_for(inst, frame_idx, cam_name):
    from colors import get_track_color
    ident_id = session.get_identity_id_for_track(frame_idx, cam_name, inst.track_idx)
    ident = session.get_identity(ident_id)
    if ident is not None:
        return ident.color
    return get_track_color(inst.track_idx)

def show_frame_grid(frame_idx, panels=window._video_panels, max_cols=4):
    cams = session.camera_names()
    n = len(cams)
    cols = min(max_cols, n)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4.5 * cols, 3.4 * rows), squeeze=False)
    fg = session.frame_group(frame_idx)
    skel = session.skeleton

    for k, cam in enumerate(cams):
        ax = axes[k // cols][k % cols]
        ax.set_title(f"{cam}  f={frame_idx}", fontsize=9)
        ax.axis("off")
        dec = panels.get(cam)._decoder if panels.get(cam) else None
        if dec is not None:
            qimg = dec.get_frame(frame_idx)
            if qimg is not None:
                ax.imshow(qimage_to_numpy(qimg))
        if fg is None:
            continue
        for inst in fg.instances.get(cam, []):
            c = color_for(inst, frame_idx, cam)
            pts = inst.points
            for a, b in skel.edges:
                if pts[a] is not None and pts[b] is not None:
                    (xa, ya), (xb, yb) = pts[a], pts[b]
                    ax.plot([xa, xb], [ya, yb], "-", color=c, linewidth=1.2)
            xs = [p[0] for p in pts if p is not None]
            ys = [p[1] for p in pts if p is not None]
            ax.scatter(xs, ys, s=10, color=c, edgecolors="k", linewidths=0.4)
            if pts[0] is not None:
                ident_id = session.get_identity_id_for_track(frame_idx, cam, inst.track_idx)
                ident = session.get_identity(ident_id)
                label = ident.name if ident else f"t{inst.track_idx}"
                ax.annotate(label, pts[0], color="white", fontsize=7,
                            xytext=(4, -4), textcoords="offset points",
                            bbox=dict(facecolor=c, edgecolor="none", pad=1, alpha=0.85))
    for k in range(n, rows * cols):
        axes[k // cols][k % cols].axis("off")
    plt.tight_layout()
    plt.show()

show_frame_grid(FRAME0)

## 2. Frame 1 — with priors

Now feed `prev_assignments` (from frame 0) and `prev_targets3d` (3D anchors). Two things change vs frame 0:

1. **Epipolar/reprojection scoring** gets a `+0.3` bonus whenever a candidate `(cam1.trackA, cam2.trackB)` was previously co-assigned to the same identity. Continuity nudges Hungarian toward last frame's pairing.
2. **`reorder_groups_by_prev`** runs: a Hungarian over a 4-signal cost where every prior 3D target competes for every current group. Signals are
   - exp(-mean reprojection distance / 50 px)
   - exp(-mean 3D-keypoint distance / 30 units)
   - mean exp(-cross-view OKS error / 50 px)
   - **2× track-continuity** (fraction of cams in the group whose `(cam, track)` already mapped to that prior identity)
   The track-continuity term is the main reason identities stay stuck to the right animal once they're labeled — even if geometry jitters.

In [ ]:
FRAME1 = FRAME0 + 1
fg1 = session.frame_group(FRAME1)

r1 = lt.match_frame_instances(
    fg1, session.cameras, session,
    num_animals=NUM_ANIMALS,
    prev_assignments=r0["assignments"],
    prev_targets3d=r0["targets3d"],
    per_frame=True,
    caches=caches,
)

print(f"frame {FRAME1}")
print(f"  groups found:      {len(r1['groups'])}")
print(f"  group sizes:       {[len(g) for g in r1['groups']]}")
print(f"  assignments (n={len(r1['assignments'])}):")
for k, v in sorted(r1['assignments'].items()):
    print(f"    {k:>10s} -> id={v}")

In [ ]:
window.set_current_frame(FRAME1)
show_frame_grid(FRAME1)

## 3. Propagation check — did identities stay glued to the right animal?

For every `(cam, track)` that received an identity in frame 0, see what happens in frame 1. Any row where `id_0 != id_1` is an **identity swap** that downstream consumers (timeline, exports) would have to deal with.

In [ ]:
import pandas as pd

rows = []
keys = sorted(set(r0["assignments"]) | set(r1["assignments"]))
for k in keys:
    cam, tr = k.split(":")
    rows.append({
        "cam": cam,
        "track_idx": int(tr),
        "id_frame0": r0["assignments"].get(k),
        "id_frame1": r1["assignments"].get(k),
        "changed": (r0["assignments"].get(k) is not None
                    and r1["assignments"].get(k) is not None
                    and r0["assignments"].get(k) != r1["assignments"].get(k)),
    })
df = pd.DataFrame(rows).sort_values(["track_idx", "cam"]).reset_index(drop=True)
n_changed = int(df["changed"].sum())
print(f"{n_changed} of {len(df)} (cam, track) bindings changed identity between frame {FRAME0} and {FRAME1}")
df

## 4. Optional — run a short range, then the full sweep

### 4a. Manual loop over a range (so you can inspect each `result`)

In [ ]:
RANGE = range(FRAME0, FRAME0 + 20)

# Re-wipe to start clean
session.identities = []
session.track_identity_map = {}
session.frame_identity_map = {}
session._identity_counter = 0
session.identities_changed.emit()
session.identity_map_changed.emit()

caches = lt.TrackerCaches()
prev_a, prev_t = None, None
history = []

for fi in RANGE:
    fg = session.frame_group(fi)
    if fg is None:
        continue
    r = lt.match_frame_instances(
        fg, session.cameras, session,
        num_animals=NUM_ANIMALS,
        prev_assignments=prev_a, prev_targets3d=prev_t,
        per_frame=True, caches=caches,
    )
    history.append({"frame": fi, **r})
    if r["assignments"]:
        prev_a = r["assignments"]
    if r["targets3d"]:
        prev_t = r["targets3d"]

swap_count = 0
for a, b in zip(history[:-1], history[1:]):
    for k, v in a["assignments"].items():
        if k in b["assignments"] and b["assignments"][k] != v:
            swap_count += 1
print(f"processed {len(history)} frames; {swap_count} (cam,track,frame) identity swaps")

### 4b. Full `track_all` sweep (mirrors clicking **Track All** in the web app)

Wipes `session.identities` on entry (set `clear_existing=False` to keep them). Returns a list of per-frame results. Progress callback updates a status string every 50 frames.

In [ ]:
from time import perf_counter

def on_progress(k, fi, result):
    if k % 50 == 0:
        print(f"  ... {k:5d} frames done  (frame {fi}, {result['num_identities']} ids)")

t0 = perf_counter()
history = lt.track_all(session,
                       num_animals=NUM_ANIMALS,
                       per_frame=True,
                       on_progress=on_progress,
                       clear_existing=True)
elapsed = perf_counter() - t0
print(f"track_all: {len(history)} frames in {elapsed:.1f}s  ({len(history)/max(elapsed,1e-6):.1f} fps)")
print("identities:", [(i.id, i.name, i.color) for i in session.identities])

# Tell the GUI to repaint with the new identity colors
session.set_color_mode("identity")
window.update()

### 4c. Identity occupancy heatmap

Quick visual of how stable the assignments were across the whole sweep — row = identity, column = frame, cell = number of cameras with that identity at that frame.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ids = [i.id for i in session.identities]
id_to_row = {i: r for r, i in enumerate(ids)}
frames = [h["frame"] for h in history]
occ = np.zeros((len(ids), len(frames)), dtype=int)
for col, h in enumerate(history):
    counts = {}
    for v in h["assignments"].values():
        counts[v] = counts.get(v, 0) + 1
    for ident_id, c in counts.items():
        if ident_id in id_to_row:
            occ[id_to_row[ident_id], col] = c

fig, ax = plt.subplots(figsize=(12, 1 + 0.4 * len(ids)))
im = ax.imshow(occ, aspect="auto", interpolation="nearest", cmap="viridis")
ax.set_yticks(range(len(ids)))
ax.set_yticklabels([session.get_identity(i).name for i in ids])
ax.set_xlabel("frame index (within history)")
ax.set_title("# cameras assigned each identity per frame")
plt.colorbar(im, ax=ax, label="n_cams")
plt.tight_layout(); plt.show()

## Reference — what the helper exports

```
luc3d_tracker_helper
├─ Geometry
│  ├─ rodrigues(rvec) -> R
│  ├─ projection_matrix(cam) -> P (3,4)
│  ├─ compute_fundamental_matrix(cam_a, cam_b) -> F (3,3)
│  ├─ undistort_points(uv, cam) -> ideal pixels
│  ├─ triangulate_dlt(obs, Ps) -> (3,) | None
│  ├─ reproject_point(X, P), reproject_points(Xs, P)
│  └─ instance_pixel_distance(reproj, points)
│
├─ Scoring
│  ├─ epipolar_score(instA, camA, instB, camB, caches)
│  ├─ reprojection_score(...)            # OKS, σ=20 px
│  └─ cross_view_score(...)              # 0.4*epi + 0.6*reproj
│
├─ Core
│  ├─ collect_instances(frame_group, cameras)
│  ├─ match_pairwise(...)                # cold-start bootstrap
│  ├─ reorder_groups_by_prev(...)        # temporal Hungarian
│  ├─ triangulate_group(group, cam_map, caches)
│  ├─ match_frame_instances(fg, cameras, session, *,
│  │       num_animals, prev_assignments, prev_targets3d,
│  │       per_frame=True, caches=None)
│  └─ track_all(session, *, num_animals, per_frame=True,
│       on_progress=None, clear_existing=True)
│
└─ TrackerCaches                         # F/P/undistort cache; pass through frames
```

### Gotchas
1. `track_all` **wipes `session.identities`** by default (matches JS `trackAll`). Pass `clear_existing=False` to keep manual labels.
2. Identity *creation order* on frame 0 = descending cross-view confidence. If you want a specific identity-to-track mapping, call `session.add_identity(name, color)` for each animal *before* running, then pass `num_animals=N`; the helper tops up only if there aren't enough.
3. `match_frame_instances` writes per-frame *and* global identity maps every call. The lucid-lite GUI repaints on `identity_map_changed`, which fires from each `set_frame_identity`.
4. Reuse one `TrackerCaches` across frames — F-matrix construction is the most expensive per-frame piece, and the JS does the same.